# mask mAP de los 4 segmentadores de la etapa 1 (U-Net++, DeepLabV3+, SegFormer-B2, YOLO11m-seg)
**Settings:** GPU T4, **Internet ON** (config de SegFormer y `ultralytics`). Solo inferencia, sin re-entrenar.
**Inputs:** salida de `benchmark-segmentacion-cv-nino` (pesos `results/<modelo>/fold<k>_weights_fp16.pt` y `fold<k>_best.pt`, `metricas_ic95.csv`) y salida de `datasets-limpios-icdas-segmentacion` (630 fotos, máscaras FDI, manifiesto con `child_id` y fold).

Especificación: `claude/encargo_map_segmentadores.md`. Preprocesamiento, `predict_native`, labelmap Universal→FDI y mapeo de clases de YOLO copiados de `benchmark-segmentacion-cv-nino`.

**Regla de instancias (principal).** Referencia: una instancia por número FDI en cada foto (todos sus píxeles). Semánticos: una instancia por clase presente en el argmax a resolución nativa (≥ 50 px, el mismo umbral de "clase predicha" del benchmark); score = softmax medio de la clase sobre sus píxeles. YOLO: instancias tal cual (`conf=0.001`, `iou=0.7`, `max_det=300`, `retina_masks=True`).
**Evaluador:** pycocotools (máscaras RLE; también cajas como secundario), `maxDets=100`, área "all", 101 puntos de recall. Mismo evaluador para los 4 modelos.

In [ ]:
!pip -q install segmentation-models-pytorch ultralytics pycocotools
import os, glob, json, time, zipfile, re, gc, pickle, gzip, copy, contextlib, io
import numpy as np, pandas as pd, cv2, torch, torch.nn as nn, torch.nn.functional as F
import segmentation_models_pytorch as smp
from transformers import SegformerForSemanticSegmentation
from pycocotools import mask as MU
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from concurrent.futures import ThreadPoolExecutor
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print(torch.__version__, smp.__version__, DEV, torch.cuda.get_device_name(0) if DEV == 'cuda' else '')
MODELS = ['unetpp_r50', 'deeplabv3p_r50', 'segformer_b2', 'yolo11m_seg']
LABEL = {'unetpp_r50': 'U-Net++', 'deeplabv3p_r50': 'DeepLabV3+', 'segformer_b2': 'SegFormer-B2', 'yolo11m_seg': 'YOLO11m-seg'}
SEMANTIC = MODELS[:3]
RECIPES = {'unetpp_r50': dict(kind='smp', arch='UnetPlusPlus', encoder='resnet50'), 'deeplabv3p_r50': dict(kind='smp', arch='DeepLabV3Plus', encoder='resnet50'),
           'segformer_b2': dict(kind='hf', ckpt='nvidia/mit-b2'), 'yolo11m_seg': dict(kind='yolo', imgsz=640)}
SZ, YSZ = 512, 640
MIN_PX = 50            # área mínima de una instancia semántica (umbral de "clase predicha" del benchmark)
YOLO_ARGS = dict(conf=0.001, iou=0.7, max_det=300, retina_masks=True)
PAINT_CONF = 0.25      # máscara pintada de YOLO del benchmark (chequeo de sanidad y S3)
B, SEED = 2000, 2026
TOL = 0.002
IN, W, TMP = '/kaggle/input', '/kaggle/working', '/kaggle/tmp'
OUT = f'{W}/map'; os.makedirs(OUT, exist_ok=True); os.makedirs(TMP, exist_ok=True)
T0 = time.time()

## Datos, labelmap y FDI (igual que en el benchmark)

In [ ]:
def find(name):
    h = sorted(glob.glob(f'{IN}/**/{name}', recursive=True)); return h[0] if h else None
man = pd.read_csv(find('segmentation_manifest.csv'))
DATA = f'{TMP}/seg'
if not glob.glob(f'{DATA}/**/{man.image_file[0]}', recursive=True):
    zipfile.ZipFile(find('fdi_segmentation_curated.zip')).extractall(DATA)
ROOT = os.path.dirname(os.path.dirname(glob.glob(f'{DATA}/**/{man.image_file[0]}', recursive=True)[0]))
LM = (glob.glob(f'{DATA}/**/labelmap.txt', recursive=True) or [find('labelmap.txt')])[0]
lines = [l.strip() for l in open(LM, encoding='utf-8') if l.strip() and not l.startswith('#')]
NAMES, COLORS = [], []
for l in lines:
    p = l.split(':'); NAMES.append(p[0].strip()); COLORS.append(tuple(int(v) for v in p[1].split(',')) if len(p) > 1 and p[1] else None)
bg = [i for i, n in enumerate(NAMES) if n.lower() in ('background', 'fondo', 'bg')]
if bg and bg[0] != 0:
    i = bg[0]; NAMES.insert(0, NAMES.pop(i)); COLORS.insert(0, COLORS.pop(i))
NC = len(NAMES)
m0 = cv2.imread(os.path.join(ROOT, man.mask_file[0]), cv2.IMREAD_UNCHANGED); RGB_MASK = m0.ndim == 3
LUT = {c: i for i, c in enumerate(COLORS) if c is not None}
def read_mask(rel):
    m = cv2.imread(os.path.join(ROOT, rel), cv2.IMREAD_UNCHANGED)
    if not RGB_MASK: return m.astype(np.uint8)
    m = cv2.cvtColor(m[..., :3], cv2.COLOR_BGR2RGB); key = (m[..., 0].astype(np.int32) << 16) | (m[..., 1].astype(np.int32) << 8) | m[..., 2]
    out = np.zeros(m.shape[:2], np.uint8); seen = set(np.unique(key))
    for c, i in LUT.items():
        k = (c[0] << 16) | (c[1] << 8) | c[2]
        if k in seen: out[key == k] = i
    return out
with ThreadPoolExecutor(8) as ex:
    IMGS = dict(zip(man.image_id, ex.map(lambda r: cv2.cvtColor(cv2.imread(os.path.join(ROOT, r)), cv2.COLOR_BGR2RGB), man.image_file)))
    MASKS = dict(zip(man.image_id, ex.map(read_mask, man.mask_file)))
def univ(n):
    m = re.search(r'\d+', n); return int(m.group()) if m else None
def to_fdi(u):
    if u is None or not 1 <= u <= 32: return None
    if u <= 8: return 10 + (9 - u)
    if u <= 16: return 20 + (u - 8)
    if u <= 24: return 30 + (25 - u)
    return 40 + (u - 24)
U = [univ(n) for n in NAMES]; FDI = [to_fdi(u) if i else None for i, u in enumerate(U)]
assert NC == 33 and all(f is not None for f in FDI[1:])
TEETH = list(range(1, NC))
assert man.groupby('child_id').fold.nunique().max() == 1
print(len(IMGS), 'fotos |', man.child_id.nunique(), 'niños | clases', NC, '| raíz', ROOT)
print(man.groupby('fold').agg(fotos=('image_id', 'size'), ninos=('child_id', 'nunique')).T.to_string())
RES = os.path.dirname(find('metricas_ic95.csv')); print('pesos y métricas del benchmark:', RES)
REF = pd.read_csv(f'{RES}/metricas_ic95.csv').set_index('model')
print(REF[['dice_tooth', 'id_f1']].round(3).to_string())

## Referencia COCO: una instancia por FDI y foto
Las fotos llevan un id entero en el orden del manifiesto; `category_id` = número FDI.

In [ ]:
IMG_IDS = {j: n + 1 for n, j in enumerate(man.image_id)}
ID2IMG = {v: k for k, v in IMG_IDS.items()}
FOLD = dict(zip(man.image_id, man.fold)); CHILD = dict(zip(man.image_id, man.child_id))
CATS = sorted(FDI[1:])
def rle(b):
    r = MU.encode(np.asfortranarray(b.astype(np.uint8))); return r
GT = {}
for j in man.image_id:
    g = MASKS[j]; GT[j] = {FDI[c]: rle(g == c) for c in np.unique(g) if c}
n_ref = sum(len(v) for v in GT.values())
print('instancias de referencia:', n_ref, f'({n_ref / len(GT):.1f} por foto)')
def coco_gt():
    imgs = [dict(id=IMG_IDS[j], height=MASKS[j].shape[0], width=MASKS[j].shape[1]) for j in man.image_id]
    anns = []
    for j, d in GT.items():
        for c, r in d.items():
            anns.append(dict(id=len(anns) + 1, image_id=IMG_IDS[j], category_id=int(c), segmentation=r, area=float(MU.area(r)),
                             bbox=MU.toBbox(r).tolist(), iscrowd=0))
    C = COCO(); C.dataset = dict(images=imgs, annotations=anns, categories=[dict(id=int(c), name=str(c)) for c in CATS])
    with contextlib.redirect_stdout(io.StringIO()): C.createIndex()
    return C
CGT = coco_gt()

## Re-inferencia en los folds de prueba
- Semánticos: igual que `predict_native` (entrada 512, autocast, logits bilineales a la resolución de la máscara), más softmax. Por foto se guardan: argmax (para el chequeo), una instancia por clase con score medio (principal) y máximo (S1), y una instancia por componente conexo (S2).
- YOLO: `conf=0.001` (principal) y, aparte, `conf=0.25` pintado por confianza creciente como en el benchmark (chequeo y S3). Imágenes recreadas igual que en el entrenamiento (foto llevada al tamaño de la máscara, JPEG calidad 95).
- Durante la inferencia se acumula, por foto, la matriz de confusión de píxeles para el chequeo de Dice y F1.

In [ ]:
class HFSeg(nn.Module):
    def __init__(self, ckpt):
        super().__init__(); self.m = SegformerForSemanticSegmentation.from_pretrained(ckpt, num_labels=NC, ignore_mismatched_sizes=True)
    def forward(self, x): return F.interpolate(self.m(pixel_values=x).logits, size=x.shape[-2:], mode='bilinear', align_corners=False)
def build(r):
    if r['kind'] == 'smp': return getattr(smp, r['arch'])(encoder_name=r['encoder'], encoder_weights=None, classes=NC)
    return HFSeg(r['ckpt'])
MEAN, STD = np.array([0.485, 0.456, 0.406], np.float32), np.array([0.229, 0.224, 0.225], np.float32)

def confusion(g, p):
    return np.bincount(g.ravel().astype(np.int64) * NC + p.ravel(), minlength=NC * NC).reshape(NC, NC)

def sem_instances(am, pa):
    """am: argmax (h, w) uint8; pa: prob. softmax de la clase ganadora en cada píxel (h, w) float32."""
    main, comp = [], []
    cnt = np.bincount(am.ravel(), minlength=NC); s = np.bincount(am.ravel(), weights=pa.ravel(), minlength=NC)
    mx = np.zeros(NC); np.maximum.at(mx, am.ravel(), pa.ravel())
    for c in np.nonzero(cnt)[0]:
        if c == 0 or cnt[c] < MIN_PX: continue
        b = am == c; main.append((FDI[c], float(s[c] / cnt[c]), float(mx[c]), rle(b)))
        n, lab = cv2.connectedComponents(b.astype(np.uint8), connectivity=8)
        if n == 2: comp.append((FDI[c], float(s[c] / cnt[c]), main[-1][3])); continue
        lc = np.bincount(lab[b], minlength=n); ls = np.bincount(lab[b], weights=pa[b], minlength=n)
        for q in range(1, n):
            if lc[q] >= MIN_PX: comp.append((FDI[c], float(ls[q] / lc[q]), rle(lab == q)))
    return main, comp

@torch.no_grad()
def infer_sem(net, ids, bs=4):
    net.eval(); out = {}
    for i in range(0, len(ids), bs):
        b = ids[i:i + bs]
        x = torch.stack([torch.from_numpy(((cv2.resize(IMGS[j], (SZ, SZ)).astype(np.float32) / 255 - MEAN) / STD).transpose(2, 0, 1)) for j in b]).to(DEV)
        with torch.autocast('cuda', enabled=DEV == 'cuda'): lg = net(x).float()
        for j, l in zip(b, lg):
            h, w = MASKS[j].shape
            l = F.interpolate(l[None], size=(h, w), mode='bilinear', align_corners=False)[0]
            pr = l.softmax(0); am = l.argmax(0)
            pa = pr.gather(0, am[None])[0]
            am, pa = am.byte().cpu().numpy(), pa.cpu().numpy()
            main, comp = sem_instances(am, pa)
            out[j] = dict(main=main, comp=comp, cm=confusion(MASKS[j], am))
    return out

def yolo_images():
    d = f'{TMP}/yolo/images'; os.makedirs(d, exist_ok=True)
    for j in man.image_id:
        f = f'{d}/{j}.jpg'
        if not os.path.exists(f):
            h, w = MASKS[j].shape
            cv2.imwrite(f, cv2.cvtColor(cv2.resize(IMGS[j], (w, h)), cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, 95])
    return d

def infer_yolo(model, ids, d):
    out = {}
    for j in ids:
        h, w = MASKS[j].shape
        # principal: instancias tal cual
        res = model.predict(f'{d}/{j}.jpg', imgsz=YSZ, verbose=False, device=0, **YOLO_ARGS)[0]
        inst = []
        if res.masks is not None:
            ms = res.masks.data; cl = res.boxes.cls.cpu().numpy().astype(int); cf = res.boxes.conf.cpu().numpy()
            if ms.shape[-2:] != (h, w): ms = F.interpolate(ms[None].float(), size=(h, w), mode='nearest')[0]
            ms = (ms > 0.5).cpu().numpy()
            inst = [(int(model.names[c]), float(s), rle(m)) for c, s, m in zip(cl, cf, ms)]
            del ms
        # benchmark: conf 0.25 pintado por confianza creciente (+ mapa de confianza del píxel para S3)
        res = model.predict(f'{d}/{j}.jpg', imgsz=YSZ, conf=PAINT_CONF, retina_masks=True, verbose=False, device=0)[0]
        p = np.zeros((h, w), np.uint8); pc = np.zeros((h, w), np.float32); n25 = 0
        if res.masks is not None:
            ms = res.masks.data.cpu().numpy(); cl = res.boxes.cls.cpu().numpy().astype(int); cf = res.boxes.conf.cpu().numpy(); n25 = len(cf)
            for q in np.argsort(cf):
                mm = cv2.resize(ms[q], (w, h), interpolation=cv2.INTER_NEAREST) > 0.5; p[mm] = cl[q] + 1; pc[mm] = cf[q]
        cnt = np.bincount(p.ravel(), minlength=NC); s = np.bincount(p.ravel(), weights=pc.ravel(), minlength=NC)
        s3 = [(FDI[c], float(s[c] / cnt[c]), rle(p == c)) for c in np.nonzero(cnt)[0] if c]
        out[j] = dict(main=inst, s3=s3, cm=confusion(MASKS[j], p), n25=n25,
                      dup=sum(v > 1 for v in pd.Series([x[0] for x in inst]).value_counts()) if inst else 0)
    return out

In [ ]:
CACHE = f'{OUT}/instancias.pkl.gz'
prev = find('instancias.pkl.gz')
if prev and not os.path.exists(CACHE): import shutil; shutil.copy2(prev, CACHE)
INST = pickle.load(gzip.open(CACHE)) if os.path.exists(CACHE) else {m: {} for m in MODELS}
from ultralytics import YOLO
yd = yolo_images()
for k in range(5):
    te = man[man.fold == k].image_id.tolist()
    for m in MODELS:
        if all(j in INST[m] for j in te): continue
        t = time.time()
        if m in SEMANTIC:
            net = build(RECIPES[m])
            net.load_state_dict({n: v.float() if v.is_floating_point() else v for n, v in torch.load(f'{RES}/{m}/fold{k}_weights_fp16.pt', map_location='cpu').items()})
            INST[m].update(infer_sem(net.to(DEV), te)); del net
        else:
            model = YOLO(f'{RES}/{m}/fold{k}_best.pt')
            names = {int(i): str(n) for i, n in model.names.items()}
            ok = names == {c - 1: str(FDI[c]) for c in TEETH}
            print(f'  YOLO fold {k}: mapeo clase→FDI {"coincide" if ok else "NO COINCIDE"} con data.yaml (names del checkpoint: 0→{names[0]}, 31→{names[31]})')
            assert ok, 'mapeo de clases de YOLO distinto del data.yaml del entrenamiento'
            INST[m].update(infer_yolo(model, te, yd)); del model
        gc.collect(); torch.cuda.empty_cache()
        print(f'{m} fold {k}: {len(te)} fotos, {time.time() - t:.0f} s')
    pickle.dump(INST, gzip.open(CACHE, 'wb'))
print('inferencia', round((time.time() - T0) / 60, 1), 'min')

## Chequeo de sanidad: Dice por diente y F1 de identificación contra `metricas_ic95.csv`
Mismas definiciones del benchmark: Dice por clase sumando sobre fotos y promediado sobre dientes; identificado = diente de la referencia con Dice ≥ 0,5; VPP = clases predichas (≥ 50 px) con Dice ≥ 0,5. Tolerancia ±0,002. **Si no coincide, el notebook se detiene aquí.**

In [ ]:
def sanity(m):
    I = np.zeros(NC); S = np.zeros(NC); ident = []; tpp = []
    for j in man.image_id:
        cm = INST[m][j]['cm']; inter = np.diag(cm).astype(float); ng = cm.sum(1); npx = cm.sum(0)
        for c in TEETH:
            if ng[c] == 0 and npx[c] < 50: continue
            d = 2 * inter[c] / max(1, ng[c] + npx[c]); I[c] += inter[c]; S[c] += ng[c] + npx[c]
            if ng[c] > 0: ident.append(d >= 0.5)
            if npx[c] >= 50: tpp.append(d >= 0.5)
    keep = S[1:] > 0; dice = (2 * I[1:][keep] / S[1:][keep]).mean()
    se, pp = np.mean(ident), np.mean(tpp); return dice, 2 * se * pp / (se + pp)
chk = []
for m in MODELS:
    d, f = sanity(m); chk.append(dict(modelo=LABEL[m], dice_tooth=d, dice_ref=REF.loc[m, 'dice_tooth'], id_f1=f, id_f1_ref=REF.loc[m, 'id_f1']))
chk = pd.DataFrame(chk); chk['dif_dice'] = chk.dice_tooth - chk.dice_ref; chk['dif_f1'] = chk.id_f1 - chk.id_f1_ref
chk['ok'] = (chk.dif_dice.abs() <= TOL) & (chk.dif_f1.abs() <= TOL)
chk.to_csv(f'{OUT}/chequeo_sanidad.csv', index=False); print(chk.round(4).to_string(index=False))
assert chk.ok.all(), 'LA RE-INFERENCIA NO REPRODUCE EL BENCHMARK: se detiene (ver chequeo_sanidad.csv)'
print('chequeo OK: la re-inferencia reproduce Dice y F1 a ±0,002')

## Detecciones por escenario y conteos
- `main`: regla principal (semánticos: score medio; YOLO: instancias tal cual).
- `S1`: semánticos con score máximo. `S2`: semánticos con componentes conexos. `S3`: YOLO pintado (conf ≥ 0,25) con la regla semántica, score = confianza media ponderada por píxeles. `S4`: YOLO con el mismo umbral de área que los semánticos (≥ 50 px). En cada escenario, los modelos que no cambian usan la regla principal.

In [ ]:
def dets(m, sc='main'):
    D = {}
    for j, r in INST[m].items():
        if m in SEMANTIC:
            if sc == 'S1': D[j] = [(c, mx, q) for c, _, mx, q in r['main']]
            elif sc == 'S2': D[j] = r['comp']
            else: D[j] = [(c, s, q) for c, s, _, q in r['main']]
        elif sc == 'S3': D[j] = r['s3']
        elif sc == 'S4': D[j] = [x for x in r['main'] if MU.area(x[2]) >= MIN_PX]
        else: D[j] = r['main']
    return D
cnt = []
for m in MODELS:
    D = dets(m); n = sum(len(v) for v in D.values())
    row = dict(modelo=LABEL[m], instancias_ref=n_ref, instancias_pred=n, por_foto=n / len(D))
    if m in SEMANTIC: row['instancias_S2'] = sum(len(v) for v in dets(m, 'S2').values())
    else:
        row['instancias_conf025'] = sum(r['n25'] for r in INST[m].values()); row['instancias_S3'] = sum(len(v) for v in dets(m, 'S3').values())
        row['instancias_menores_50px'] = sum(MU.area(x[2]) < MIN_PX for r in INST[m].values() for x in r['main']); row['fotos_con_duplicados'] = sum(r['dup'] > 0 for r in INST[m].values()); row['pares_foto_FDI_duplicados'] = sum(r['dup'] for r in INST[m].values())
    cnt.append(row)
cnt = pd.DataFrame(cnt); cnt.to_csv(f'{OUT}/conteo_instancias.csv', index=False); print(cnt.round(1).to_string(index=False))

## Evaluación con pycocotools (combinada fuera de fold y por fold)

In [ ]:
THR = np.linspace(.5, .95, 10)
def coco_eval(D, iou_type='segm', imgs=None):
    res = [dict(image_id=IMG_IDS[j], category_id=int(c), segmentation=q, score=float(s)) for j, L in D.items() for c, s, q in L]
    with contextlib.redirect_stdout(io.StringIO()):
        dt = CGT.loadRes(res); E = COCOeval(CGT, dt, iou_type)
        E.params.imgIds = sorted(IMG_IDS[j] for j in (imgs if imgs is not None else D)); E.params.maxDets = [100]
        E.params.areaRng = [[0, 1e10]]; E.params.areaRngLbl = ['all']
        E.evaluate(); E.accumulate()
    P = E.eval['precision'][:, :, :, 0, 0]            # (T, R, K)
    valid = (P > -1).all(axis=(0, 1))
    ap = np.where(valid, P.clip(min=0).mean(1), np.nan)   # (T, K) AP por umbral y clase
    return dict(mAP50=np.nanmean(ap[0]), mAP75=np.nanmean(ap[5]), mAP50_95=np.nanmean(ap.mean(0)), n_clases=int(valid.sum()),
                ap_cls=ap, pr50=P[0][:, valid].mean(1), cats=E.params.catIds)
CO, FO = {}, []
for m in MODELS:
    D = dets(m)
    for it in ('segm', 'bbox'):
        t = time.time(); CO[(m, it)] = coco_eval(D, it)
        for k in range(5):
            r = coco_eval({j: v for j, v in D.items() if FOLD[j] == k}, it)
            FO.append(dict(modelo=LABEL[m], tipo='mask' if it == 'segm' else 'box', fold=k, **{x: r[x] for x in ('mAP50', 'mAP75', 'mAP50_95', 'n_clases')}))
        print(f'{LABEL[m]:13s} {it}: mAP50 {CO[(m, it)]["mAP50"]:.3f}  mAP75 {CO[(m, it)]["mAP75"]:.3f}  mAP50–95 {CO[(m, it)]["mAP50_95"]:.3f}  '
              f'({CO[(m, it)]["n_clases"]} clases, {time.time() - t:.0f} s con folds)')
FO = pd.DataFrame(FO); FO.to_csv(f'{OUT}/map_por_fold.csv', index=False)

## AP propio (para el bootstrap) y verificación contra pycocotools
Réplica de COCOeval: por foto y clase, detecciones ordenadas por score (orden estable) y truncadas a 100; emparejamiento greedy (con una instancia de referencia por clase y foto, es la primera detección con IoU ≥ umbral); acumulación por clase con precisión envolvente y 101 puntos de recall. Las fotos repetidas en un remuestreo cuentan como fotos distintas (pesos enteros por niño).

In [ ]:
def match_table(D, iou_type='segm'):
    """Una fila por detección: clase, foto, score, TP en los 10 umbrales. Orden = el de COCOeval."""
    cat, img, sc, tp = [], [], [], []
    for j in sorted(D, key=lambda x: IMG_IDS[x]):
        L = D[j]
        for c in sorted({x[0] for x in L}):
            Lc = [x for x in L if x[0] == c]
            o = np.argsort([-x[1] for x in Lc], kind='mergesort')[:100]; Lc = [Lc[i] for i in o]
            t = np.zeros((len(Lc), 10), bool)
            if c in GT[j]:
                if iou_type == 'segm': iou = MU.iou([x[2] for x in Lc], [GT[j][c]], [0]).ravel()
                else: iou = MU.iou([MU.toBbox(x[2]) for x in Lc], [MU.toBbox(GT[j][c])], [0]).ravel()
                for ti, th in enumerate(THR):
                    h = np.nonzero(iou >= min(th, 1 - 1e-10))[0]
                    if len(h): t[h[0], ti] = True
            cat += [c] * len(Lc); img += [j] * len(Lc); sc += [x[1] for x in Lc]; tp.append(t)
    return dict(cat=np.array(cat), img=np.array(img, dtype=object), score=np.array(sc, float), tp=np.vstack(tp) if tp else np.zeros((0, 10), bool))
NGT = pd.DataFrame([(j, c) for j, d in GT.items() for c in d], columns=['img', 'cat'])
KIDS = np.array(sorted(man.child_id.unique())); KID_IX = {c: i for i, c in enumerate(KIDS)}
RT = np.linspace(0, 1, 101)

def ap_weighted(M, Wk):
    """M: tabla de emparejamiento; Wk: (nb, n_niños) multiplicidad de cada niño. Devuelve AP (nb, 10, K) con NaN si la clase no tiene referencia."""
    nb = Wk.shape[0]; out = np.full((nb, 10, len(CATS)), np.nan)
    det_kid = np.array([KID_IX[CHILD[j]] for j in M['img']], int)
    gk = NGT.groupby('cat').img.apply(lambda s: np.bincount([KID_IX[CHILD[j]] for j in s], minlength=len(KIDS)))
    for ki, c in enumerate(CATS):
        npig = Wk @ gk[c] if c in gk.index else np.zeros(nb)
        sel = np.nonzero(M['cat'] == c)[0]
        if len(sel) == 0: out[:, :, ki] = np.where(npig[:, None] > 0, 0.0, np.nan); continue
        o = sel[np.argsort(-M['score'][sel], kind='mergesort')]
        w = Wk[:, det_kid[o]].astype(float)                    # (nb, nd)
        nd = w.shape[1]; base = 2 * np.arange(nb)[:, None]
        for ti in range(10):
            t = M['tp'][o, ti]
            tpc = np.cumsum(w * t, 1); fpc = np.cumsum(w * ~t, 1)
            with np.errstate(invalid='ignore', divide='ignore'):
                rc = tpc / npig[:, None]; pr = tpc / (tpc + fpc)
            pr = np.nan_to_num(pr); pr = np.maximum.accumulate(pr[:, ::-1], 1)[:, ::-1]
            rc = np.nan_to_num(rc, nan=0.0, posinf=0.0)
            flat = (rc + base).ravel(); qy = (RT[None, :] + base).ravel()
            idx = np.searchsorted(flat, qy, side='left').reshape(nb, 101)
            loc = idx - np.arange(nb)[:, None] * nd
            q = np.where(loc < nd, pr.ravel()[np.minimum(idx, nb * nd - 1)], 0.0)
            out[:, ti, ki] = np.where(npig > 0, q.mean(1), np.nan)
    return out
def summ(ap):   # ap: (nb, 10, K)
    return dict(mAP50=np.nanmean(ap[:, 0], 1), mAP75=np.nanmean(ap[:, 5], 1), mAP50_95=np.nanmean(np.nanmean(ap, 1), 1))
MT, ver = {}, []
ONE = np.ones((1, len(KIDS)), int)
for m in MODELS:
    for it in ('segm', 'bbox'):
        MT[(m, it)] = match_table(dets(m), it); own = ap_weighted(MT[(m, it)], ONE); s = summ(own); r = CO[(m, it)]
        dmax = np.nanmax(np.abs(own[0] - r['ap_cls']))
        ver.append(dict(modelo=LABEL[m], tipo=it, **{f'{x}_coco': r[x] for x in ('mAP50', 'mAP75', 'mAP50_95')}, **{f'{x}_propio': s[x][0] for x in ('mAP50', 'mAP75', 'mAP50_95')},
                        max_dif_AP_clase=dmax))
ver = pd.DataFrame(ver)
ver['coincide_3dec'] = [all(round(r[f'{x}_coco'], 3) == round(r[f'{x}_propio'], 3) or abs(r[f'{x}_coco'] - r[f'{x}_propio']) < 5e-4 for x in ('mAP50', 'mAP75', 'mAP50_95')) for _, r in ver.iterrows()]
ver.to_csv(f'{OUT}/verificacion_ap_propio.csv', index=False); print(ver.round(4).to_string(index=False))
assert ver.coincide_3dec.all(), 'el AP propio no coincide con pycocotools: no se corre el bootstrap'

## IC 95 % por bootstrap de niños (2000, semilla 2026), resumen y comparación pareada (Holm)
Los mismos remuestreos para los 4 modelos (pareado).

In [ ]:
rng = np.random.default_rng(SEED)
DRAWS = rng.integers(0, len(KIDS), size=(B, len(KIDS)))
WK = np.zeros((B, len(KIDS)), np.int32)
for b in range(B): WK[b] = np.bincount(DRAWS[b], minlength=len(KIDS))
BOOT = {}
for key, M in MT.items():
    t = time.time(); BOOT[key] = summ(np.concatenate([ap_weighted(M, WK[i:i + 250]) for i in range(0, B, 250)])); print(LABEL[key[0]], key[1], f'{time.time() - t:.0f} s')
res = []
for m in MODELS:
    for it in ('segm', 'bbox'):
        r = CO[(m, it)]; tipo = 'mask' if it == 'segm' else 'box'; f = FO[(FO.modelo == LABEL[m]) & (FO.tipo == tipo)]
        row = dict(modelo=LABEL[m], tipo=tipo, n_clases=r['n_clases'])
        for x in ('mAP50', 'mAP75', 'mAP50_95'):
            row[x] = r[x]; row[f'{x}_lo'] = np.quantile(BOOT[(m, it)][x], .025); row[f'{x}_hi'] = np.quantile(BOOT[(m, it)][x], .975)
            row[f'{x}_fold_media'] = f[x].mean(); row[f'{x}_fold_de'] = f[x].std()
        res.append(row)
res = pd.DataFrame(res); res.to_csv(f'{OUT}/map_resumen.csv', index=False)
for tipo in ('mask', 'box'):
    print(f'\n{tipo} mAP — combinado fuera de fold [IC 95 %] | por fold media ± DE')
    for _, r in res[res.tipo == tipo].iterrows():
        print(f"{r.modelo:13s} " + '  '.join(f"{x} {r[x]:.3f} [{r[x + '_lo']:.3f}–{r[x + '_hi']:.3f}] | {r[x + '_fold_media']:.3f} ± {r[x + '_fold_de']:.3f}" for x in ('mAP50', 'mAP75', 'mAP50_95')))
cmp = []
for it in ('segm', 'bbox'):
    for x in ('mAP50', 'mAP50_95'):
        part = []
        for a in range(4):
            for b_ in range(a + 1, 4):
                ma, mb = MODELS[a], MODELS[b_]; d = BOOT[(ma, it)][x] - BOOT[(mb, it)][x]
                part.append(dict(tipo='mask' if it == 'segm' else 'box', metrica=x, a=LABEL[ma], b=LABEL[mb], dif=CO[(ma, it)][x] - CO[(mb, it)][x],
                                 lo=np.quantile(d, .025), hi=np.quantile(d, .975), p=min(1, 2 * min((d <= 0).mean(), (d >= 0).mean()))))
        part = pd.DataFrame(part).sort_values('p').reset_index(drop=True); n = len(part)
        part['p_holm'] = np.minimum(1, np.maximum.accumulate([(n - i) * p for i, p in enumerate(part.p)])); cmp.append(part)
cmp = pd.concat(cmp); cmp.to_csv(f'{OUT}/comparacion_pareada_map.csv', index=False)
print('\nComparación pareada (a − b), bootstrap de niños, Holm dentro de cada métrica y tipo:'); print(cmp.round(3).to_string(index=False))

## AP por diente (combinado, mask y box)

In [ ]:
apd = []
nref = NGT.groupby('cat').size()
for m in MODELS:
    for it in ('segm', 'bbox'):
        r = CO[(m, it)]
        for ki, c in enumerate(r['cats']):
            apd.append(dict(modelo=LABEL[m], tipo='mask' if it == 'segm' else 'box', diente=c, n_ref=int(nref.get(c, 0)), AP50=r['ap_cls'][0, ki], AP75=r['ap_cls'][5, ki], AP50_95=np.nanmean(r['ap_cls'][:, ki])))
apd = pd.DataFrame(apd); apd.to_csv(f'{OUT}/ap_por_diente.csv', index=False)
w = apd[apd.tipo == 'mask'].pivot(index=['diente', 'n_ref'], columns='modelo', values=['AP50', 'AP50_95'])
print(w.round(3).to_string())

## Curvas precisión–sensibilidad a IoU 0,5 (mask, todas las clases promediadas)

In [ ]:
fig, ax = plt.subplots(figsize=(5.2, 4.4))
for m, col in zip(MODELS, ['#4C72B0', '#55A868', '#C44E52', '#8172B2']):
    ax.plot(RT, CO[(m, 'segm')]['pr50'], color=col, lw=1.8, label=f"{LABEL[m]} (AP50 {CO[(m, 'segm')]['mAP50']:.3f})")
ax.set_xlabel('Sensibilidad (recall)'); ax.set_ylabel('Precisión'); ax.set_xlim(0, 1); ax.set_ylim(0, 1.02); ax.grid(alpha=.3)
ax.set_title('Máscaras, IoU 0,5 (media sobre dientes FDI)', fontsize=10); ax.legend(loc='lower left', fontsize=8, frameon=False)
fig.tight_layout(); fig.savefig(f'{OUT}/curvas_pr_iou50.pdf'); fig.savefig(f'{OUT}/curvas_pr_iou50.png', dpi=200); plt.show()

## Sensibilidad de la regla de instancias (mask, combinado)

In [ ]:
sen = []
for sc in ('main', 'S1', 'S2', 'S3', 'S4'):
    for m in MODELS:
        changed = (sc in ('S1', 'S2') and m in SEMANTIC) or (sc in ('S3', 'S4') and m not in SEMANTIC)
        r = coco_eval(dets(m, sc)) if changed else CO[(m, 'segm')]
        sen.append(dict(escenario=sc, modelo=LABEL[m], cambia=changed or sc == 'main', mAP50=r['mAP50'], mAP75=r['mAP75'], mAP50_95=r['mAP50_95']))
sen = pd.DataFrame(sen)
for x in ('mAP50', 'mAP50_95'): sen[f'orden_{x}'] = sen.groupby('escenario')[x].rank(ascending=False).astype(int)
sen.to_csv(f'{OUT}/map_sensibilidad.csv', index=False)
print(sen.pivot(index='modelo', columns='escenario', values=['mAP50', 'mAP50_95']).round(3).to_string())
print(sen.pivot(index='modelo', columns='escenario', values=['orden_mAP50', 'orden_mAP50_95']).to_string())
print('\ntiempo total', round((time.time() - T0) / 60, 1), 'min'); print(sorted(os.listdir(OUT)))